In [1]:
# ============================================================
# audit-s7-axis — STANDALONE fixed-anchor CLAP axis notebook.
# ONE cell. Does NOT touch audit-score (which stays frozen/canonical).
#
# Setup (once): new Kaggle notebook, GPU on, Internet ON.
# Inputs to attach: audit-generate-A, audit-generate-C, audit-generate-D
#   (notebook outputs), audit-generate-b-output (dataset, loose sao wavs),
#   audit-s6-output (dataset: scored_full_v2.csv + s7_bootstrap_cis.csv).
# Then Save & Run All. ~12-15 min total. Banks:
#   s7_clap_axis.csv, s7_axis_level_means.csv, s7_bootstrap_cis.csv
#   (input copy + axis rows merged in — drop-in replacement everywhere).
#
# S7A.0 is a PROBE: embeds the 10 anchor texts + 1 s of silence and
# prints full type/shape diagnostics BEFORE touching the corpus. If the
# transformers API surprises us again, this dies inside ~2 minutes with
# everything needed to fix it from the log.
# Axis embedding logic is DIMENSION-DRIVEN: uses returned embeddings
# as-is when already joint-space; projects through the model's own
# projection heads only when encoder-width. Deterministic: no fusion,
# first-10 s window, eval, no_grad, seed 0.
# ============================================================
import os, glob, time, tarfile, shutil
import numpy as np, pandas as pd
from scipy.stats import spearmanr

S7_B, S7_SEED = 2000, 0
S7_CKPT = 'laion/larger_clap_music'      # fallback below if unavailable
SRC, DUR = 48000, 10.0

# ---------- S7A.0 CLAP load + instrumented probe ----------
print('='*72 + '\nS7A.0  CLAP PROBE\n' + '='*72)
import torch, librosa
import transformers as _tfm
from transformers import ClapModel, ClapProcessor
torch.manual_seed(0)
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  transformers {getattr(_tfm, "__version__", "?")} | torch {getattr(torch, "__version__", "?")} | {_dev}')

def _load_reported(name):
    """Load model+processor; print weight-loading integrity report."""
    try:
        _m, _info = ClapModel.from_pretrained(name, output_loading_info=True)
        miss, unex = _info.get('missing_keys', []), _info.get('unexpected_keys', [])
        mism = _info.get('mismatched_keys', [])
        print(f'    load report: missing {len(miss)}, unexpected {len(unex)}, mismatched {len(mism)}')
        if miss: print('      first missing:', miss[:3])
        if unex: print('      first unexpected:', unex[:3])
        if len(miss) > 5:
            print('      *** WEIGHTS PARTIALLY RANDOM (key mismatch between checkpoint'
                  ' and this transformers version) ***')
    except TypeError:
        _m = ClapModel.from_pretrained(name)
        print('    load report unavailable (no output_loading_info kwarg)')
    return _m.to(_dev).eval(), ClapProcessor.from_pretrained(name)

def _head_dims(head):
    lins = [m for m in head.modules() if isinstance(m, torch.nn.Linear)]
    return lins[0].in_features, lins[-1].out_features

def _cp_audio(batch):
    """transformers renamed the kwarg (audios= -> audio=); try both."""
    try:
        return _cp(audio=batch, sampling_rate=SRC, return_tensors='pt', padding=True)
    except TypeError:
        return _cp(audios=batch, sampling_rate=SRC, return_tensors='pt', padding=True)

def _describe(out, label):
    print(f'  [{label}] return type: {type(out).__name__}')
    if isinstance(out, torch.Tensor):
        print(f'    tensor shape {tuple(out.shape)}')
    else:
        for a in dir(out):
            if a.startswith('_'): continue
            v = getattr(out, a, None)
            if isinstance(v, torch.Tensor):
                print(f'    .{a}: Tensor {tuple(v.shape)}')

def _extract(out, head, in_dim, out_dim, label):
    """Version-proof: prefer explicit embeds attrs; else tensor; else
    pooled output. Project ONLY if the vector is encoder-width."""
    t = None
    if isinstance(out, torch.Tensor):
        t = out
    else:
        for a in ('text_embeds', 'audio_embeds', 'pooler_output'):
            v = getattr(out, a, None)
            if isinstance(v, torch.Tensor): t = v; break
        if t is None and isinstance(out, (tuple, list)) and len(out):
            t = out[1] if len(out) > 1 else out[0]
    assert isinstance(t, torch.Tensor), f'{label}: no tensor in {type(out).__name__}'
    d = t.shape[-1]
    if d == out_dim:
        pass                          # already joint-space
    elif d == in_dim:
        t = head(t)                   # encoder-width -> project (4.x-faithful)
    else:
        raise AssertionError(f'{label}: dim {d} matches neither encoder '
                             f'({in_dim}) nor joint ({out_dim}) width')
    return t.detach().float().cpu().numpy()

def _fwd_embeds(text_inputs, audio_inputs):
    """Joint embeddings via the full contrastive forward — the path the
    checkpoint was trained through. Returns (text_embeds, audio_embeds) np."""
    kw = {k: v.to(_dev) for k, v in text_inputs.items()}
    kw.update({k: v.to(_dev) for k, v in audio_inputs.items()})
    with torch.no_grad():
        out = _cm(**kw)
    te = getattr(out, 'text_embeds', None); ae = getattr(out, 'audio_embeds', None)
    assert te is not None and ae is not None, \
        f'forward output lacks embeds: {type(out).__name__}'
    return (te.detach().float().cpu().numpy(), ae.detach().float().cpu().numpy())

def _cosmat(E):
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    return E @ E.T

def _mat_report(name, E):
    C = _cosmat(E); off = C[~np.eye(len(C), dtype=bool)]
    wg = [C[i, i+1] for i in range(0, len(C), 2)]        # within-genre lo-hi
    print(f'    [{name}] off-diag cos: min {off.min():+.4f}  mean {off.mean():+.4f}'
          f'  max {off.max():+.4f} | within-genre lo-hi: '
          + ' '.join(f'{c:+.3f}' for c in wg)
          + ('   << DEGENERATE (all anchors ~identical)' if _degenerate(E) else '   OK'))

def _degenerate(anchor_embeds):
    """True if the 10 anchor embeddings are effectively one vector."""
    C = _cosmat(anchor_embeds)
    off = C[~np.eye(len(C), dtype=bool)]
    return float(off.min()) > 0.999

_ANCHOR_TEXTS = None   # set in S7A.1 before candidate validation
_probe_texts = ['instrumental rock track, very calm',
                'instrumental rock track, extremely intense']

def _np_of(t):
    try: return t.detach().cpu().numpy()
    except Exception: return np.asarray(getattr(t, 'arr', t))

def _validate_candidate(name):
    """Load a checkpoint, verify tokenization + weight integrity, and test
    anchor separation on both extraction paths. Returns (cm, cp, method,
    anchors_np) if healthy, else None."""
    global _cm, _cp, _t_in, _t_out, _a_in, _a_out
    print(f'\n  CANDIDATE: {name}')
    try:
        _cm, _cp = _load_reported(name)
    except Exception as e:
        print(f'    load FAILED: {type(e).__name__}: {e}'); return None
    _t_in, _t_out = _head_dims(_cm.text_projection)
    _a_in, _a_out = _head_dims(_cm.audio_projection)
    ti2 = _cp(text=_probe_texts, return_tensors='pt', padding=True)
    ids = _np_of(ti2['input_ids'])
    print(f'    input_ids[0][:12]: {list(ids[0][:12])}')
    print(f'    input_ids[1][:12]: {list(ids[1][:12])}')
    print('    tokenization ' + ('OK (texts tokenize differently)'
          if not np.array_equal(ids[0], ids[1]) else 'BROKEN (identical ids!)'))
    tiA = _cp(text=_ANCHOR_TEXTS, return_tensors='pt', padding=True)
    with torch.no_grad():
        eA = _extract(_cm.get_text_features(**{k: v.to(_dev) for k, v in tiA.items()}),
                      _cm.text_projection, _t_in, _t_out, 'anchors')
    _mat_report('get_*_features path', eA)
    eB = None
    try:
        eB, _ = _fwd_embeds(tiA, _cp_audio([np.zeros(int(SRC*1.0), np.float32)]))
        _mat_report('full-forward path ', eB)
    except Exception as fe:
        print(f'    forward-path unavailable: {type(fe).__name__}: {fe}')
    if not _degenerate(eA): return _cm, _cp, 'features', eA
    if eB is not None and not _degenerate(eB): return _cm, _cp, 'forward', eB
    print('    candidate degenerate on both paths — trying next')
    return None

# ---------- S7A.1 data + corpus + axis over 1,500 clips ----------
print('\n' + '='*72 + '\nS7A.1  AXIS OVER CORPUS\n' + '='*72)
_sfp = sorted(glob.glob('/kaggle/input/**/scored_full_v2.csv', recursive=True))
assert _sfp, 'attach the audit-s6-output dataset (scored_full_v2.csv not found)'
S7 = pd.read_csv(_sfp[0]); assert len(S7) == 1500
print(f'  scores: {_sfp[0]}')
_gt = S7.drop_duplicates('genre_slug').set_index('genre_slug')['genre_text'].to_dict()
_anch = {g: (f'{t}, very calm', f'{t}, extremely intense') for g, t in _gt.items()}
for g, (lo, hi) in sorted(_anch.items()): print(f'    {g}: "{lo}" | "{hi}"')

_texts, _tkey = [], []
for g, (a_lo, a_hi) in sorted(_anch.items()):
    _texts += [a_lo, a_hi]; _tkey += [(g, 'lo'), (g, 'hi')]
_ANCHOR_TEXTS = _texts
print('\n  ANCHOR VALIDATION — candidate checkpoints until one separates anchors')
_sel = None
for _cand in [S7_CKPT, 'laion/clap-htsat-unfused', 'laion/clap-htsat-fused']:
    _sel = _validate_candidate(_cand)
    if _sel is not None:
        _cm, _cp, _METHOD, _te = _sel
        print(f'\n  SELECTED: {_cand} via {_METHOD} path')
        break
if _sel is None:
    raise SystemExit('ALL candidates degenerate — paste this whole block, plus the '
                     'CLAP cell from the audit-score notebook above')
_te = _te / np.linalg.norm(_te, axis=1, keepdims=True)
_tix = {k: i for i, k in enumerate(_tkey)}
print(f'  {len(_texts)} anchor embeddings ready ({_te.shape[1]}-d, method={_METHOD})')

_axp = 's7_clap_axis.csv'
_cands = ([_axp] if os.path.exists(_axp) else []) + \
         sorted(glob.glob('/kaggle/input/**/s7_clap_axis.csv', recursive=True))
_ax = pd.DataFrame(columns=['file','sim_lo','sim_hi','axis'])
for _cnd in _cands:
    try:
        _prev = pd.read_csv(_cnd)
        if len(_prev) > len(_ax): _ax = _prev; print(f'  resuming from cache: {_cnd} ({len(_prev)} rows)')
    except Exception: pass
_ax = _ax[_ax['file'].isin(S7['file'])]
if len(_ax) and float(pd.to_numeric(_ax['axis']).std()) < 1e-3:
    print(f'  cached axis is DEGENERATE (std {pd.to_numeric(_ax["axis"]).std():.2e})'
          ' — discarding cache, recomputing fresh')
    _ax = pd.DataFrame(columns=['file','sim_lo','sim_hi','axis'])
_todo = [f for f in S7['file'] if f not in set(_ax['file'])]
if _todo:
    _pf, _C, _mk = {}, '/tmp/s7_corpus', False
    _nb = {os.path.basename(f): f for f in _todo}
    for root in ['/kaggle/input']:
        for p in glob.glob(f'{root}/**/*.wav', recursive=True):
            b = os.path.basename(p)
            if b in _nb: _pf[_nb.pop(b)] = p
    for t in sorted(glob.glob('/kaggle/input/**/*.tar', recursive=True) +
                    glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)):
        if not _nb: break
        try:
            if not _mk: os.makedirs(_C, exist_ok=True); _mk = True
            with tarfile.open(t, 'r:*') as tf:
                n0 = len(_nb)
                for mem in tf.getmembers():
                    b = os.path.basename(mem.name)
                    if mem.isfile() and b in _nb:
                        fo = tf.extractfile(mem)
                        if fo is None: continue
                        dst = os.path.join(_C, b)
                        with open(dst, 'wb') as out: out.write(fo.read())
                        _pf[_nb.pop(b)] = dst
                print(f'    {os.path.basename(t)}: pulled {n0-len(_nb)} clips')
        except Exception as te:
            print(f'    tar skip {os.path.basename(t)}: {te}')
    print(f'  resolved audio for {len(_pf)}/{len(_todo)} clips')
    _g_of = S7.set_index('file')['genre_slug'].to_dict()
    _m_of = S7.set_index('file')['model'].to_dict()
    _fails = []
    import warnings as _wn
    def _load10(p):
        with _wn.catch_warnings():
            _wn.simplefilter('ignore')
            try:
                y, _ = librosa.load(p, sr=SRC, mono=True, duration=DUR)  # FIRST 10 s
            except Exception:                        # backend fallback
                import soundfile as _sf
                data, sr0 = _sf.read(p, dtype='float32', always_2d=True)
                y = data.mean(axis=1)[:int(sr0*DUR)]
                y = librosa.resample(y, orig_sr=sr0, target_sr=SRC)
        out = np.zeros(int(SRC*DUR), dtype=np.float32)
        out[:min(len(y), len(out))] = y[:len(out)]
        return out
    # PREFLIGHT: one clip per model through the loader, verdict printed now
    print('  loader preflight (one clip per model):')
    for m in sorted(S7['model'].unique()):
        f0 = S7[S7['model'] == m]['file'].iloc[0]
        p0 = _pf.get(f0)
        try:
            _ = _load10(p0); print(f'    {m:5s} OK   ({os.path.basename(str(p0))})')
        except Exception as e:
            print(f'    {m:5s} FAIL {type(e).__name__}: {e}')
    _ti1 = _cp(text=[_texts[0]], return_tensors='pt', padding=True)
    _new, _bF, _bY, t0 = [], [], [], time.time()
    def _flush():
        if not _bF: return
        _aiB = _cp_audio([y for y in _bY])
        if _METHOD == 'forward':
            _, em = _fwd_embeds(_ti1, _aiB)
        else:
            with torch.no_grad():
                em = _extract(_cm.get_audio_features(**{k: v.to(_dev) for k, v in _aiB.items()}),
                              _cm.audio_projection, _a_in, _a_out, 'audio-batch')
        em = em / np.linalg.norm(em, axis=1, keepdims=True)
        for f, e in zip(_bF, em):
            g = _g_of[f]
            s_lo = float(e @ _te[_tix[(g,'lo')]]); s_hi = float(e @ _te[_tix[(g,'hi')]])
            _new.append(dict(file=f, sim_lo=s_lo, sim_hi=s_hi, axis=s_hi-s_lo))
        _bF.clear(); _bY.clear()
    for k, f in enumerate(_todo, 1):
        p = _pf.get(f)
        if p is None: continue
        try:
            _bF.append(f); _bY.append(_load10(p))
        except Exception as e:
            if _bF and _bF[-1] == f: _bF.pop()
            _fails.append(dict(file=f, model=_m_of.get(f, '?'),
                               stage='load', error=f'{type(e).__name__}: {e}'))
        if len(_bF) == 16: _flush()
        if k == 4:
            _flush(); print('  SMOKE (first clips):')
            for r in _new[:4]: print(f"    {r['file']}: axis {r['axis']:+.4f}")
        if k % 200 == 0: print(f'    {k}/{len(_todo)}  ({time.time()-t0:.0f}s)')
    _flush()
    _ax = pd.DataFrame(_new) if _ax.empty else pd.concat([_ax, pd.DataFrame(_new)],
                                                          ignore_index=True)
    for c in ['sim_lo','sim_hi','axis']: _ax[c] = pd.to_numeric(_ax[c])
    _ax.to_csv(_axp, index=False)
    print(f'  cached -> {_axp} ({len(_ax)}/1500 clips)')
    if _fails:
        _fdf = pd.DataFrame(_fails); _fdf.to_csv('s7_axis_failures.csv', index=False)
        print('  FAILURE TABLE (banked s7_axis_failures.csv):')
        print(_fdf.groupby('model').size().to_string())
        for _, r in _fdf.drop_duplicates('model').iterrows():
            print(f'    first {r.model}: {r.error[:140]}')
    if _mk: shutil.rmtree(_C, ignore_errors=True)

# ---------- S7A.2 stats (seed-0 bootstraps, identical helpers) ----------
print('\n' + '='*72 + '\nS7A.2  AXIS STATS\n' + '='*72)
def _ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if a.size == 0: return float('nan'), float('nan')
    return np.percentile(a, 2.5), np.percentile(a, 97.5)
def _brho(lv, sc, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([spearmanr(lv[i], sc[i])[0] for i in idx])
def _bstat(arrs, fn, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    return np.array([fn([a[ix[b]] for a, ix in zip(arrs, idxs)]) for b in range(B)])

_cov = len(_ax) / 1500
if _cov < 0.40:
    raise SystemExit(f'axis coverage too low to report ({len(_ax)}/1500) — '
                     'read the FAILURE TABLE above and s7_axis_failures.csv')
if _cov < 0.90:
    print(f'  *** PARTIAL COVERAGE ({len(_ax)}/1500) — stats below are '
          'DIAGNOSTIC ONLY, do not cite; fix failures and rerun ***')
for c in ['sim_lo','sim_hi','axis']: _ax[c] = pd.to_numeric(_ax[c])
_A = S7.merge(_ax, on='file')
_M = sorted(S7['model'].unique())
print('  per-model coverage: ' + ', '.join(
    f"{m} {(_A['model']==m).sum()}/300" for m in _M))
_rows = []
_PARTIAL_NOTE = '' if _cov >= 0.90 else f'PARTIAL coverage {len(_ax)}/1500'
def _bank(analysis, mdl, scope, point, boots, note=''):
    note = note or _PARTIAL_NOTE
    lo, hi = _ci(boots)
    _rows.append(dict(analysis=analysis, model=mdl, scope=scope,
                      point=round(float(point), 4), lo=round(float(lo), 4),
                      hi=round(float(hi), 4), note=note))
    return lo, hi

print('  axis means by level (verbal):')
print(_A[_A['phrasing']=='verbal'].groupby(['model','level'])['axis'].mean().unstack().round(3).to_string())
_A.groupby(['model','phrasing','level'])['axis'].mean().round(4).to_csv('s7_axis_level_means.csv')
print('\n  rho(level, axis) with 95% CI:')
for m in _M:
    for ph in ['numeric','verbal']:
        g = _A[(_A['model']==m)&(_A['phrasing']==ph)]
        if len(g) < 10:
            print(f'    {m:5s} {ph:8s} SKIPPED (coverage {len(g)}/150 — see failure table)')
            continue
        r = spearmanr(g['level'], g['axis'])[0]
        lo, hi = _bank('rho_level_axis', m, ph, r, _brho(g['level'].values, g['axis'].values))
        print(f'    {m:5s} {ph:8s} {r:+.3f} [{lo:+.3f},{hi:+.3f}]')
print('\n  axis droop (verbal L9 - L7), fourth witness:')
for m in _M:
    g7 = _A[(_A['model']==m)&(_A['phrasing']=='verbal')&(_A['level']==7)]['axis'].values
    g9 = _A[(_A['model']==m)&(_A['phrasing']=='verbal')&(_A['level']==9)]['axis'].values
    if len(g7) < 3 or len(g9) < 3:
        print(f'    {m:5s} SKIPPED (coverage L7 {len(g7)}/30, L9 {len(g9)}/30)')
        continue
    pt = g9.mean()-g7.mean()
    lo, hi = _bank('droop_L9-L7_axis', m, 'verbal', pt,
                   _bstat([g7,g9], lambda xs: xs[1].mean()-xs[0].mean()))
    v = 'DROPS' if hi < 0 else ('rises' if lo > 0 else 'CI incl. 0')
    print(f'    {m:5s} {pt:+.4f} [{lo:+.4f},{hi:+.4f}]  {v}')

# ---------- S7A.3 merge + bank ----------
_cip = sorted(glob.glob('/kaggle/input/**/s7_bootstrap_cis.csv', recursive=True))
if _cip: print(f'  merge base: {_cip[0]}')
_base = pd.read_csv(_cip[0]) if _cip else pd.DataFrame(
    columns=['analysis','model','scope','point','lo','hi','note'])
_base = _base[~_base['analysis'].isin(['rho_level_axis','droop_L9-L7_axis'])]
pd.concat([_base, pd.DataFrame(_rows)], ignore_index=True) \
  .to_csv('s7_bootstrap_cis.csv', index=False)
print('\n' + '='*72)
print('S7A BANKED: s7_clap_axis.csv, s7_axis_level_means.csv,')
print(f'  s7_bootstrap_cis.csv (merged: {len(_base)} prior rows + {len(_rows)} axis rows).')
print('  Standalone axis complete — audit-score untouched.')

S7A.0  CLAP PROBE
  transformers 5.0.0 | torch 2.10.0+cu128 | cuda

S7A.1  AXIS OVER CORPUS
  scores: /kaggle/input/datasets/anon/scoring-output/scored_full_v2.csv
    edm: "instrumental electronic dance track, very calm" | "instrumental electronic dance track, extremely intense"
    folk: "instrumental acoustic folk track, very calm" | "instrumental acoustic folk track, extremely intense"
    hiphop: "instrumental hip-hop track, very calm" | "instrumental hip-hop track, extremely intense"
    orchestral: "instrumental orchestral track, very calm" | "instrumental orchestral track, extremely intense"
    rock: "instrumental rock track, very calm" | "instrumental rock track, extremely intense"

  ANCHOR VALIDATION — candidate checkpoints until one separates anchors

  CANDIDATE: laion/larger_clap_music


config.json:   0%|          | 0.00/628 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/776M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/776M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

    input_ids[0][:12]: [np.int64(0), np.int64(179), np.int64(41392), np.int64(337), np.int64(3152), np.int64(1349), np.int64(6), np.int64(182), np.int64(6327), np.int64(2)]
    input_ids[1][:12]: [np.int64(0), np.int64(179), np.int64(41392), np.int64(337), np.int64(3152), np.int64(1349), np.int64(6), np.int64(2778), np.int64(5676), np.int64(2)]
    tokenization OK (texts tokenize differently)
    [get_*_features path] off-diag cos: min +0.2627  mean +0.6287  max +0.8755 | within-genre lo-hi: +0.632 +0.817 +0.809 +0.771 +0.816   OK
    [full-forward path ] off-diag cos: min +0.2627  mean +0.6287  max +0.8755 | within-genre lo-hi: +0.632 +0.817 +0.809 +0.771 +0.816   OK

  SELECTED: laion/clap-htsat-unfused via features path
  10 anchor embeddings ready (512-d, method=features)
    audio_A.tar: pulled 600 clips
    audio_C.tar: pulled 300 clips
    audio_D.tar: pulled 300 clips
  resolved audio for 1500/1500 clips
  loader preflight (one clip per model):
    ace   OK   (ace__edm_L1_num__